# 01 — Data Exploration**Master's Thesis — Heterogeneous Effects of AI Tool Adoption on Developer Job Satisfaction**This notebook is a first pass at understanding the 2025 Stack Overflow Developer Survey. The goal here is *not* to clean or model anything yet — just to see what we have: how many rows and columns, what the key variables look like, and where the missingness is.We focus early attention on the variables that matter for the thesis:- **Treatment:** AI tool adoption (`AISelect`)- **Outcome:** Job satisfaction (`JobSat`)- **Key moderators:** experience (`WorkExp`, `YearsCode`), organization size (`OrgSize`), role (`DevType`)- **Covariates:** age, education, country, industry, remote work, compensation

## Setup

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snspd.set_option("display.max_columns", 200)pd.set_option("display.max_rows", 200)sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)

## Load the dataThe raw survey lives in `data/raw/`. We load both the responses and the schema (the schema maps each column name to the full question text, which is very handy for a survey this wide).

In [ ]:
RAW = "../data/raw/"df = pd.read_csv(RAW + "survey_results_public.csv", low_memory=False)schema = pd.read_csv(RAW + "survey_results_schema.csv", encoding="utf-8-sig")print(f"Responses: {df.shape[0]:,} rows  ×  {df.shape[1]} columns")print(f"Schema:    {schema.shape[0]} question entries")

### A small helper: look up what a column meansWith ~170 columns, it's easy to lose track of what each one is. This helper prints the full question text for any column.

In [ ]:
def question(colname):    """Return the full question text for a given column name."""    match = schema.loc[schema["qname"] == colname, "question"]    if len(match):        return match.iloc[0]    return "(not found in schema)"# Examplefor col in ["AISelect", "JobSat", "WorkExp", "OrgSize", "DevType"]:    print(f"{col:12s} → {question(col)}")

## First look at the data

In [ ]:
df.head()

In [ ]:
df.info(verbose=False)

## MissingnessSurvey data is missing-heavy by design — most questions are optional, and the survey randomizes some blocks. Let's see how complete each column is, then zoom in on the variables we actually care about.

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)missing_df = missing.to_frame("pct_missing").round(1)print("Most incomplete columns:")missing_df.head(20)

In [ ]:
print("Most complete columns:")missing_df.tail(20)

## The variables that matter for the thesisLet's isolate the columns that map onto our research design and check their completeness specifically.

In [ ]:
key_vars = {    "Treatment":  ["AISelect", "AISent", "AIAcc", "AIComplex", "AIThreat", "AIAgents"],    "Outcome":    ["JobSat"],    "Moderators": ["WorkExp", "YearsCode", "OrgSize", "DevType"],    "Covariates": ["Age", "EdLevel", "Country", "Industry", "RemoteWork",                   "ICorPM", "CompTotal"],}rows = []for group, cols in key_vars.items():    for c in cols:        if c in df.columns:            rows.append({                "group": group,                "column": c,                "pct_missing": round(df[c].isna().mean() * 100, 1),                "n_unique": df[c].nunique(),                "question": question(c)[:70],            })key_summary = pd.DataFrame(rows)key_summary

## The treatment variable: AI tool adoption`AISelect` is the question we'll use to define adopters vs. non-adopters. Let's see the actual response categories and their distribution.

In [ ]:
print(question("AISelect"))print()df["AISelect"].value_counts(dropna=False)

In [ ]:
ax = df["AISelect"].value_counts().plot(kind="barh")ax.set_title("AI tool adoption (AISelect)")ax.set_xlabel("Number of respondents")plt.tight_layout()plt.show()

## The outcome: job satisfaction`JobSat` is our primary outcome. Let's check whether it's numeric or categorical and see its distribution.

In [ ]:
print(question("JobSat"))print()print(df["JobSat"].describe())print()df["JobSat"].value_counts(dropna=False).sort_index()

In [ ]:
if pd.api.types.is_numeric_dtype(df["JobSat"]):    ax = df["JobSat"].plot(kind="hist", bins=20)    ax.set_title("Job satisfaction (JobSat)")    ax.set_xlabel("JobSat")    plt.tight_layout()    plt.show()else:    print("JobSat is not numeric — inspect the categories above.")

## Key moderatorsThese are the variables the causal forest will explore for treatment-effect heterogeneity. The experience variables are central to hypotheses H1a/H1b.

In [ ]:
print(question("WorkExp"))print()print(df["WorkExp"].describe())

In [ ]:
for col in ["OrgSize", "DevType"]:    print(f"\n{'='*60}\n{col} — {question(col)}\n{'='*60}")    print(df[col].value_counts(dropna=False).head(15))

## A first (purely descriptive) cutA quick, **non-causal** look at the raw relationship between AI adoption and job satisfaction. This is exactly the kind of naive comparison the thesis will later improve on with causal methods — but it's a useful sanity check that the variables behave sensibly.⚠️ This tells us nothing about causation yet — adopters and non-adopters differ in many ways.

In [ ]:
if pd.api.types.is_numeric_dtype(df["JobSat"]):    cut = (df.dropna(subset=["AISelect", "JobSat"])             .groupby("AISelect")["JobSat"]             .agg(["mean", "median", "count"])             .sort_values("mean", ascending=False))    print(cut.round(2))

## Notes & next stepsThings to carry into the cleaning notebook (`02-data-cleaning.ipynb`):- **Define the treatment** clearly: which `AISelect` categories count as "adopter" vs. "non-adopter"? (The proposal compares daily/weekly users against those who don't use AI and don't plan to.)- **Check the outcome scale** for `JobSat` and decide how to handle it.- **Drop irrelevant columns** — the long technology blocks (`Language*`, `Database*`, `Platform*`, `Webframe*`, `OpSys`, etc.) and Stack Overflow usage columns are not needed for this design.- **Decide on the covariate set** for confounding adjustment.- **Handle missingness** in the key variables — how many complete cases do we have once we restrict to the variables we need?*Observations so far:*> (write your own notes here as you run the cells)